In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

Enter your Account UUID: a5184defa8e847248f589d35080efbfa
Enter your API KEY: ··········


In [ ]:
NAMESPACE_PREFIX="datasciencealliance-org" # Add your namespace-prefix here

### Define Ability

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.find-events.wildfire-smoke-detection",
        description="Detect visible wildfire smoke plumes or active fire in fixed fire watch tower security camera footage",
        worker_release="qwen3-instruct",
        text_prompt="""
          Analyze the provided video footage to determine when visible wildfire smoke or active fire appears in a fixed fire watch tower security camera feed.

          The camera is a stationary outdoor surveillance camera mounted on a fire watch tower. The scene usually shows a wide landscape view of forests, dry brush, hillsides, mountain ridges, valleys, or rural terrain. The camera does not move, zoom, or pan.

          Event name: smoke_or_fire

          Label smoke_or_fire only when there is clear visual evidence of wildfire smoke or fire. This includes visible smoke plumes rising from terrain, vegetation, hillsides, forests, buildings, or open land; gray, white, brown, or dark smoke columns; smoke spreading from a ground source; visible flames; glowing fire lines; or active burning areas.

          Do not label smoke_or_fire for normal clouds, fog, mist, marine layer, haze, dust, vehicle exhaust, sunlight glare, lens flare, shadows, camera blur, or sunset colors unless there is a clear smoke plume connected to the ground or visible flame.

          Do not label smoke_or_fire just because the scene is dry, hazy, orange, low contrast, mountainous, or fire-prone. The event should only be labeled when actual smoke or fire is visible.

          Ignore all on-screen text, camera labels, fake timestamps, date stamps, battery icons, monitoring camera overlays, watermarks, captions, and UI elements. Do not read, copy, or use any timestamp printed inside the video frame as an event prediction.

          Do not label smoke_or_fire for dust clouds from vehicles, dirt roads, construction activity, wind-blown sand, or dry brush movement. Vehicle dust is usually tan or beige, low to the ground, follows a road or vehicle path, and is not connected to visible flames or an active burn source.

          Only label smoke_or_fire when there is clear visual evidence of wildfire smoke or active fire, such as smoke rising from a ground source, smoke connected to vegetation or terrain, visible flames, a glowing fire line, or an active burning area.

          If smoke or fire is visible for only part of the video, label only the timestamp range where smoke or fire is visible.

          If uncertain, do not label the event unless wildfire smoke or active fire is clearly visible.

          Return the event label smoke_or_fire only for the timestamps where visible smoke or fire appears.
          """,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=100,
            fps=1,
            image_size=640
        ),
        is_public=False
    )
]


### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

created ability 06a5531affb379398000590982b34873 with alias entries [AbilityAliasEntry(alias='datasciencealliance-org.find-events.wildfire-smoke-detection', tag='1.0.0'), AbilityAliasEntry(alias='datasciencealliance-org.find-events.wildfire-smoke-detection', tag='latest')]


### Evalulate on a Single Video

In [ ]:
from pathlib import Path
import json
import re
from collections import Counter
import cv2
from eyepop import EyePopSdk
from eyepop.worker.worker_types import InferenceComponent, Pop

pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.find-events.wildfire-smoke-detection:latest",
   )
])

video_path = "/content/wildfire_test_video.mp4"  # Add path to video
sample_video_path = Path(video_path)

# Get actual video duration
cap = cv2.VideoCapture(str(sample_video_path))
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
cap.release()

video_duration = frame_count / fps if fps and frame_count else None

positive_range_counter = Counter()
no_event_count = 0
raw_results_sample = []

negative_phrases = [
    "no event",
    "event label is not applicable",
    "not applicable",
    "no visible smoke",
    "no clear visual evidence",
    "not wildfire smoke",
    "not a wildfire",
    "vehicle dust",
    "dust kicked up",
    "dust cloud",
    "not smoke",
    "not fire"
]

def seconds_to_time(seconds):
    seconds = max(0, int(round(seconds)))
    minutes = seconds // 60
    seconds = seconds % 60
    return f"{minutes:02d}:{seconds:02d}"

def parse_event_range(text):
    text = text.strip()
    lower_text = text.lower()

    # If the model explicitly says no event / dust / not smoke, do not treat it as smoke_or_fire.
    if any(phrase in lower_text for phrase in negative_phrases):
        return None

    # Accept exact labeled event format:
    # smoke_or_fire: 00:00 - 00:04
    labeled_pattern = r"^smoke_or_fire\s*:\s*(\d{2}):(\d{2})\s*-\s*(\d{2}):(\d{2})$"
    match = re.match(labeled_pattern, text)

    # Also accept exact bare range format:
    # 00:00 - 00:04
    # This is for EyePop Find Event outputs that return only the range.
    if not match:
        bare_pattern = r"^(\d{2}):(\d{2})\s*-\s*(\d{2}):(\d{2})$"
        match = re.match(bare_pattern, text)

    if not match:
        return None

    groups = match.groups()
    start_min = int(groups[-4])
    start_sec = int(groups[-3])
    end_min = int(groups[-2])
    end_sec = int(groups[-1])

    start = start_min * 60 + start_sec
    end = end_min * 60 + end_sec

    if end <= start:
        return None

    # Reject fake camera timestamps that start after the actual video duration.
    if video_duration is not None and start > video_duration:
        return None

    # Clip end time to real video duration.
    if video_duration is not None:
        end = min(end, video_duration)

    return start, end

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   job = endpoint.upload(sample_video_path)

   while result := job.predict():
      if len(raw_results_sample) < 3:
         raw_results_sample.append(result)

      for text_item in result.get("texts", []):
         event_text = text_item.get("text", "").strip()
         lower_text = event_text.lower()

         if any(phrase in lower_text for phrase in negative_phrases):
            no_event_count += 1
            continue

         parsed = parse_event_range(event_text)
         if parsed:
            positive_range_counter[parsed] += 1

print("=== RAW RESULT SAMPLE ===")
print(json.dumps(raw_results_sample, indent=2))

print("\n=== DETECTED smoke_or_fire EVENT RANGE ===")

positive_count = sum(positive_range_counter.values())

# If the model mostly says no event, trust no event.
if no_event_count > 0 and no_event_count >= positive_count:
    print("No smoke_or_fire event detected.")
elif positive_range_counter:
    (start, end), count = positive_range_counter.most_common(1)[0]
    print(f"smoke_or_fire: {seconds_to_time(start)} - {seconds_to_time(end)}")
else:
    print("No smoke_or_fire event detected.")

if video_duration is not None:
    print(f"\nActual video duration: {video_duration:.2f} seconds")

print(f"No-event text count: {no_event_count}")
print(f"Positive range count: {positive_count}")

print("\nDone")

=== RAW RESULT SAMPLE ===
[
  {
    "duration": 41666666,
    "seconds": 0.083333333,
    "source_height": 720,
    "source_id": "7a3d3d52-7eec-11f1-bb55-52795d4727a0",
    "source_width": 1280,
    "system_timestamp": 1783968959460865000,
    "texts": [
      {
        "text": "The video shows a vehicle driving on a dirt road, kicking up a large cloud of tan dust behind it. This is consistent with vehicle exhaust/dust on a dry road, not wildfire smoke.\n\nThere is no visible smoke plume rising from the terrain, vegetation, or hillsides. The haze in the distance appears to be atmospheric haze or mist, not a ground-connected smoke column.\n\nTherefore, the event label is not applicable.\n\nNo event"
      }
    ],
    "timestamp": 83333333
  },
  {
    "duration": 41666666,
    "seconds": 0.125,
    "source_height": 720,
    "source_id": "7a3d3d52-7eec-11f1-bb55-52795d4727a0",
    "source_width": 1280,
    "system_timestamp": 1783968959461536000,
    "texts": [
      {
        "text": "